# 05. 검색 — 순수 벡터 의미검색

**무엇을 하나:** 서빙 `recipe_db`(1,693) 에 쿼리를 던져 bge-m3 벡터로 의미가 가까운 레시피를 찾는다.
`distance` 는 **낮을수록 가까움**(0에 가까울수록 유사). `1-distance` 가 유사도라고 보면 된다.

**여기서 관찰할 것 (우리 논의 핵심):**
- **구체 쿼리**(`김치찌개`, `두부 요리`)는 라벨 없이도 의미로 잘 찾는다.
- **추상/짧은 쿼리**(`한식`)는 약하다 — 의미가 빈약해 벡터가 헤매고, 소스마다 장르 라벨이 달라(식약처=`반찬/국&찌개`, 농정원=`한식`) 편향이 생긴다.
- **숫자 정렬은 못 한다** — `고단백`이라 쳐도 단백질 높은 순이 아니라 "단백질"과 의미가 가까운 순.

**벡터 + 메타 동시 필터:** `search_recipes(query, cuisine_filter=, max_time=)` 는 ChromaDB `where` 로 장르·시간을 **벡터와 함께** 거를 수 있다(이 셀에선 안 씀). 영양/재료 필터는 아직 미구현 — 입출력 계약은 `docs/rag-io-contract.md`.

In [ ]:
import sys,os,asyncio
from pathlib import Path
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B)
try: sys.stdout.reconfigure(encoding='utf-8')
except Exception: pass
from dotenv import load_dotenv; load_dotenv()
from app.rag.retriever import search_recipes
for q in ['김치찌개','두부 요리','고단백 닭가슴살','나트륨 줄인 반찬']:
    docs=asyncio.run(search_recipes(q,k=3))
    print(q,'->',[(x.get('name'),round(x.get('distance',0),3)) for x in docs])